# Validation on live data

The assignment asks for validation on real-world examples, ideally in a live development setting (D4). This notebook:

1. **Freezes** one CART and one random forest per target, trained on every run collected up to the collection date `C`, and saves them with skops.
2. **Re-tests** the saved models, unchanged, on master runs created after `C`. The models have never seen these runs, and their features are built from the stored history plus earlier new runs only.
3. **Explains** single live predictions (`predict-run`): a few correct and wrong cases, and a run that is still in progress.
4. Collects light **user feedback** on the explanations.

T2 (runs whose previous run passed) is the main target (D5). T1 is shown for comparison.

In [ ]:
import json

import numpy as np
import pandas as pd

from acise_p2_cidt.collect import http
from acise_p2_cidt.collect.gh_runs import N8N_MASTER
from acise_p2_cidt.validate import live
from acise_p2_cidt.viz import plots

REPO = N8N_MASTER.repo
TARGETS = ["T2", "T1"]
MODELS = ["cart", "forest"]
CASES_PER_OUTCOME = 1  # every predict_run does one partial build

## 1. Freeze the models

`train_final` fits on all runs up to `C` with the same protocol as one CV fold. The first 80% of runs are used for fitting. Pruning alpha and the decision threshold are chosen on the last 20%. There is no refit, so the threshold stays matched to the saved model. Hyperparameters come from the grid search on that split.

Each model is saved as `results/model/<model>_<repo>_<target>.skops`, with a `.json` sidecar next to it. The sidecar holds the features, the hyperparameters, the threshold, `C`, the training medians used to impute new runs, the code version, and the file's SHA-256. Loading checks the hash and refuses any type outside `CART`, `Tree` and `RandomForest`.

In [ ]:
paths = {}
for t in TARGETS:
    for m in MODELS:
        p = live.model_path(REPO, t, m)
        # train once: re-running the notebook must not replace the frozen model
        paths[t, m] = p if p.exists() else live.train_final(REPO, t, m)
        print(f"{t} {m}: {paths[t, m].name}")

In [ ]:
rows = []
for (t, m), p in paths.items():
    est, side = live.load_model(p)  # hash + trusted-type check
    rows.append({
        "target": t, "model": m, "cutoff (C)": side["cutoff"], "n_train": side["n_train"], "n_val": side["n_val"],
        "threshold": side["threshold"], "alpha": side["alpha"], "val_mcc": side["val_mcc"],
        "params": json.dumps(side["params"]), "code_version": side["code_version"][:12],
    })
pd.DataFrame(rows).round(4)

In [ ]:
est, side = live.load_model(paths["T2", "cart"])
print(f"depth {est.depth_}, {est.n_leaves_} leaves, features: {side['features']}\n")
print(est.export_text(max_depth=3))
plots.save(
    plots.tree_icicle(est.tree_, est.feature_names_, max_depth=3).update_layout(title="Frozen CART (T2, top 3 levels)"),
    "live_tree_icicle_T2",
)

## 2. Re-test on runs after `C`

`retest` fetches the master runs created since `C` (windows saved before their last day ended are fetched again) and fetches the new commits into the clone. It then builds features for the new runs only. History features still read every earlier run, so a new run sees the old history plus earlier new runs, never later ones. The processed table and the manifest are not touched, so the CV results stay reproducible.

Parrot (previous outcome) and majority class are scored on the same runs. The output has the same columns as `run_cv`, so the re-test can sit next to the CV folds.

In [ ]:
retests = {}
for t in TARGETS:
    frames = [live.retest(paths[t, m]) for m in MODELS]
    # parrot and majority come out of every retest call; keep one copy
    retests[t] = pd.concat(frames, ignore_index=True).drop_duplicates("method")
    display(retests[t].set_index("method")[["n_test", "n_fail", "mcc", "f1", "precision", "recall", "pct_builds_saved", "pct_failures_caught"]].round(3).rename_axis(t))

In [ ]:
for t in TARGETS:
    cv_path = live.MODEL_DIR / f"cv_results_{t}.csv"
    if not cv_path.exists():
        # tree.ipynb keeps its CV results in memory; save them there once with
        # results[t].to_csv(live.MODEL_DIR / f"cv_results_{t}.csv", index=False)
        print(f"{cv_path.name} missing: save the CV results from tree.ipynb first")
        continue
    cv = pd.read_csv(cv_path)
    cv = cv[cv.method.isin(retests[t].method)]
    display(plots.save(
        plots.retest_vs_cv(cv, retests[t]).update_layout(title=f"CV folds vs re-test on runs after C ({t})"),
        f"live_retest_vs_cv_{t}",
    ))

**Reading the figure.** A re-test point inside its CV box means the frozen model holds up on future runs. A point below the box means drift: the CI behaviour after `C` differs from the training period. The re-test covers only the days since `C`, so it has few failures, and one extra caught or missed failure moves its MCC a lot. Read it as a check, not as a precise estimate.

## 3. Case studies (`predict-run`)

Cases are taken from the T2 CART re-test, one per outcome: true positive, true negative, false positive and false negative. `predict_run` fetches each run again, builds its row, imputes it with the training medians, and prints the decision path in raw feature units. The CLI gives the same output:

    uv run predict-run <run_id|sha> --model results/model/cart_n8n-io__n8n_T2.skops

In [ ]:
def show(r: dict) -> None:
    label = {None: "running", 0: "passed", 1: "failed"}[r["label"]]
    print(f"run {r['run_id']} ({r['sha'][:10]}, {label}) {r['url']}")
    for line in r["path"] or []:
        print(f"  {line}")
    verdict = "FAIL" if r["predicted"] else "pass"
    print(f"P(fail) = {r['p_fail']:.2f} -> {verdict} at threshold {r['threshold']:.2f}")
    if not r["in_target"]:
        print("note: outside the model's target population")
    print()


path = paths["T2", "cart"]
runs = pd.read_csv(f"{path.with_suffix('')}_retest_runs.csv")
fail, pred = runs["failed"].eq(1), runs["pred"].astype(bool)
runs["outcome"] = np.select([fail & pred, ~fail & ~pred, ~fail & pred], ["TP", "TN", "FP"], "FN")
print(runs["outcome"].value_counts().to_string(), "\n")
cases = runs.groupby("outcome").head(CASES_PER_OUTCOME).sort_values("outcome")
case_results = {}
for _, c in cases.iterrows():
    print(f"=== {c.outcome}")
    case_results[c.run_id] = live.predict_run(path, str(c.run_id))
    show(case_results[c.run_id])

**Discussion (fill in after running).** For each case: which tests on the path decided it, whether those tests make sense to a developer (for example a recent failure rate, or many files changed), and, for the wrong cases, which signal the model did not have. Expected: most false negatives are flaky or environment failures, which no pre-run signal can see (see D5 and the flaky share in `collection.ipynb`).

## 4. Live: a run still in progress

This is the actual use case: predict before the outcome is known. The newest in-progress master run is predicted. Its features read only runs that completed before it was created, so the missing outcome does not matter. Its label shows as "running". Run the cell again after the run finishes to compare.

In [ ]:
running = http.get(
    f"/repos/{REPO}/actions/workflows/{N8N_MASTER.workflow_id}/runs",
    {"branch": N8N_MASTER.branch, "event": "push", "status": "in_progress", "per_page": 1},
)["workflow_runs"]
if running:
    live_run = live.predict_run(path, str(running[0]["id"]))
    show(live_run)
else:
    print("no master run in progress right now; re-run this cell later")

## 5. User feedback

5–10 explained predictions (the cases above, plus the live run) were shown to 2–3 fellow students, as decision paths only, without the actual outcome. Each person answered two questions:

- **Q1.** Is the explanation understandable? (1 = not at all, 5 = fully)
- **Q2.** Would you trust it to skip or prioritise this run? (yes / no / only with more information)

The table below records the answers.

In [ ]:
# One row per person and case; fill in by hand after the sessions.
feedback = pd.DataFrame(
    [
        # {"person": "S1", "run_id": 0, "understandable": 4, "trust": "no", "comment": ""},
    ],
    columns=["person", "run_id", "understandable", "trust", "comment"],
)
feedback.to_csv(live.MODEL_DIR / "feedback.csv", index=False)
if len(feedback):
    display(feedback.groupby("run_id")["understandable"].agg(["mean", "count"]))
    display(feedback["trust"].value_counts(normalize=True).rename("share"))

## Summary (fill in after running)

- Re-test vs CV, per target and model:
- Does the frozen tree beat Parrot and majority on unseen runs?
- What the case studies show about the explanations:
- Feedback: how understandable, and how much trust:
- Limits: a short re-test window with few failures; one repository; flaky failures cannot be predicted from pre-run data.